# KDR Hi — Model complexity analysis

This notebook analyses whether the internal validation protocol used for hyperparameter selection changes the complexity of the selected models.

The comparison is between:

- OOD holdout inner validation
- Random shuffle inner validation

The main question is:

**Does random shuffle select models that are more complex and less calibrated for final OOD test generalization?**

The analysis uses the saved per-fold artifacts:

- `params_fold_i.json`
- `complexity_fold_i.json`

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

In [2]:
PROJECT_ROOT = Path("../..").resolve()

RAW_RESULTS_DIR = PROJECT_ROOT / "results" / "hi" / "kdr"

OUTPUT_DIR = (
    PROJECT_ROOT
    / "results"
    / "results_ood_vs_random_shuffle"
    / "hi"
    / "kdr"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw results:", RAW_RESULTS_DIR)
print("Output dir:", OUTPUT_DIR)

Project root: /home/f.capria/drug-discovery-lohi
Raw results: /home/f.capria/drug-discovery-lohi/results/hi/kdr
Output dir: /home/f.capria/drug-discovery-lohi/results/results_ood_vs_random_shuffle/hi/kdr


## Experiment registry

The registry defines which result folders correspond to each combination of:

- model
- fingerprint
- protocol

In [3]:
EXPERIMENTS = [
    # Decision Tree
    {
        "model": "Decision Tree",
        "model_short": "DT",
        "fingerprint": "ECFP4",
        "protocol": "OOD holdout",
        "result_dir": "dt_kdr_hi_inner_ood_holdout_ecfp4",
    },
    {
        "model": "Decision Tree",
        "model_short": "DT",
        "fingerprint": "ECFP4",
        "protocol": "Random shuffle",
        "result_dir": "dt_kdr_hi_random_shuffle_ecfp4",
    },
    {
        "model": "Decision Tree",
        "model_short": "DT",
        "fingerprint": "MACCS",
        "protocol": "OOD holdout",
        "result_dir": "dt_kdr_hi_inner_ood_holdout_maccs",
    },
    {
        "model": "Decision Tree",
        "model_short": "DT",
        "fingerprint": "MACCS",
        "protocol": "Random shuffle",
        "result_dir": "dt_kdr_hi_random_shuffle_maccs",
    },

    # Logistic Regression
    {
        "model": "Logistic Regression",
        "model_short": "LR",
        "fingerprint": "ECFP4",
        "protocol": "OOD holdout",
        "result_dir": "lr_kdr_hi_inner_ood_holdout_ecfp4",
    },
    {
        "model": "Logistic Regression",
        "model_short": "LR",
        "fingerprint": "ECFP4",
        "protocol": "Random shuffle",
        "result_dir": "lr_kdr_hi_random_shuffle_ecfp4",
    },
    {
        "model": "Logistic Regression",
        "model_short": "LR",
        "fingerprint": "MACCS",
        "protocol": "OOD holdout",
        "result_dir": "lr_kdr_hi_inner_ood_holdout_maccs",
    },
    {
        "model": "Logistic Regression",
        "model_short": "LR",
        "fingerprint": "MACCS",
        "protocol": "Random shuffle",
        "result_dir": "lr_kdr_hi_random_shuffle_maccs",
    },
    {
        "model": "Logistic Regression",
        "model_short": "LR",
        "fingerprint": "RDKit desc",
        "protocol": "OOD holdout",
        "result_dir": "lr_kdr_hi_inner_ood_holdout_rdkit_desc",
    },
    {
        "model": "Logistic Regression",
        "model_short": "LR",
        "fingerprint": "RDKit desc",
        "protocol": "Random shuffle",
        "result_dir": "lr_kdr_hi_random_shuffle_rdkit_desc",
    },

    # Linear SVM
    {
        "model": "SVM linear",
        "model_short": "SVM",
        "fingerprint": "ECFP4",
        "protocol": "OOD holdout",
        "result_dir": "svm_linear_kdr_hi_inner_ood_holdout_ecfp4",
    },
    {
        "model": "SVM linear",
        "model_short": "SVM",
        "fingerprint": "ECFP4",
        "protocol": "Random shuffle",
        "result_dir": "svm_linear_kdr_hi_random_shuffle_ecfp4",
    },
    {
        "model": "SVM linear",
        "model_short": "SVM",
        "fingerprint": "MACCS",
        "protocol": "OOD holdout",
        "result_dir": "svm_linear_kdr_hi_inner_ood_holdout_maccs",
    },
    {
        "model": "SVM linear",
        "model_short": "SVM",
        "fingerprint": "MACCS",
        "protocol": "Random shuffle",
        "result_dir": "svm_linear_kdr_hi_random_shuffle_maccs",
    },
]

In [4]:
def read_json(path):
    with open(path, "r") as f:
        return json.load(f)


def safe_get(dct, key, default=np.nan):
    return dct.get(key, default)

In [5]:
def load_complexity_rows(experiments, raw_results_dir):
    rows = []

    for exp in experiments:
        result_dir = raw_results_dir / exp["result_dir"]

        if not result_dir.exists():
            print(f"Missing directory: {result_dir}")
            continue

        for fold in [1, 2, 3]:
            params_path = result_dir / f"params_fold_{fold}.json"
            complexity_path = result_dir / f"complexity_fold_{fold}.json"

            if not params_path.exists():
                print(f"Missing params file: {params_path}")
                continue

            if not complexity_path.exists():
                print(f"Missing complexity file: {complexity_path}")
                continue

            params = read_json(params_path)
            complexity = read_json(complexity_path)

            train_metrics = params.get("train_metrics", {})
            test_metrics = params.get("test_metrics", {})

            inner_pr_auc = params.get("inner_selection_score", np.nan)
            inner_train_pr_auc = params.get("inner_train_score", np.nan)
            train_pr_auc = train_metrics.get("pr_auc", np.nan)
            test_pr_auc = test_metrics.get("pr_auc", np.nan)

            row = {
                "model": exp["model"],
                "model_short": exp["model_short"],
                "fingerprint": exp["fingerprint"],
                "protocol": exp["protocol"],
                "result_dir": exp["result_dir"],
                "fold": fold,

                # Scores
                "inner_pr_auc": inner_pr_auc,
                "inner_train_pr_auc": inner_train_pr_auc,
                "train_pr_auc": train_pr_auc,
                "test_pr_auc": test_pr_auc,

                # Gaps
                "train_inner_gap": train_pr_auc - inner_pr_auc,
                "inner_test_gap": inner_pr_auc - test_pr_auc,
                "train_test_gap": train_pr_auc - test_pr_auc,
            }

            # Add all complexity fields
            for key, value in complexity.items():
                row[key] = value

            rows.append(row)

    return pd.DataFrame(rows)

In [6]:
complexity_all = load_complexity_rows(EXPERIMENTS, RAW_RESULTS_DIR)

complexity_all = complexity_all.sort_values(
    ["model_short", "fingerprint", "protocol", "fold"]
).reset_index(drop=True)

complexity_all.head(3)

,model,model_short,fingerprint,protocol,result_dir,fold,inner_pr_auc,inner_train_pr_auc,train_pr_auc,test_pr_auc,...,l1_norm,l2_norm,approx_margin,intercept,C,penalty,l1_ratio,dual,n_support_per_class,n_support_total
0,Decision Tree,DT,ECFP4,OOD holdout,dt_kdr_hi_inner_ood_holdout_ecfp4,1,0.862853,0.951132,0.9783,0.9105,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Decision Tree,DT,ECFP4,OOD holdout,dt_kdr_hi_inner_ood_holdout_ecfp4,2,0.785762,0.953923,0.5579,0.9438,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Decision Tree,DT,ECFP4,OOD holdout,dt_kdr_hi_inner_ood_holdout_ecfp4,3,0.902256,0.982301,0.9870,0.9726,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
print("Rows loaded:", len(complexity_all))
print("Expected rows:", len(EXPERIMENTS) * 3)

complexity_all[
    [
        "model",
        "fingerprint",
        "protocol",
        "fold",
        "inner_pr_auc",
        "train_pr_auc",
        "test_pr_auc",
        "inner_test_gap",
        "train_test_gap",
        "model_class",
    ]
]

Rows loaded: 42
Expected rows: 42


,model,fingerprint,protocol,fold,inner_pr_auc,train_pr_auc,test_pr_auc,inner_test_gap,train_test_gap,model_class
0,Decision Tree,ECFP4,OOD holdout,1,0.862853,0.9783,0.9105,-0.047647,0.0678,DecisionTreeClassifier
1,Decision Tree,ECFP4,OOD holdout,2,0.785762,0.5579,0.9438,-0.158038,-0.3859,DecisionTreeClassifier
2,Decision Tree,ECFP4,OOD holdout,3,0.902256,0.9870,0.9726,-0.070344,0.0144,DecisionTreeClassifier
3,Decision Tree,ECFP4,Random shuffle,1,0.993223,0.9925,0.5623,0.430923,0.4302,DecisionTreeClassifier
4,Decision Tree,ECFP4,Random shuffle,2,0.404528,0.5802,0.7389,-0.334372,-0.1587,DecisionTreeClassifier
5,Decision Tree,ECFP4,Random shuffle,3,0.970758,0.9797,0.6454,0.325358,0.3343,DecisionTreeClassifier
6,Decision Tree,MACCS,OOD holdout,1,0.880837,0.9918,0.9490,-0.068163,0.0428,DecisionTreeClassifier
7,Decision Tree,MACCS,OOD holdout,2,0.815139,0.5716,0.9524,-0.137261,-0.3808,DecisionTreeClassifier
8,Decision Tree,MACCS,OOD holdout,3,0.854999,0.9902,0.9355,-0.080501,0.0547,DecisionTreeClassifier
9,Decision Tree,MACCS,Random shuffle,1,0.969246,0.9769,0.5277,0.441546,0.4492,DecisionTreeClassifier


## Logistic Regression complexity table

For Logistic Regression complexity is described by:

- selected `C` - larger --> weaker regularization
- selected `l1_ratio`
- number of non-zero coefficients
- sparsity
- L1 norm of the coefficient vector
- L2 norm of the coefficient vector - 

In [8]:
lr_cols = [
    "model",
    "fingerprint",
    "protocol",
    "fold",
    "C",
    "l1_ratio",
    "class_weight",
    "n_nonzero_coefficients",
    "sparsity",
    "l1_norm",
    "l2_norm",
    "inner_pr_auc",
    "test_pr_auc",
    "inner_test_gap",
    "train_test_gap",
]

lr_table = (
    complexity_all[complexity_all["model_short"] == "LR"]
    [lr_cols]
    .sort_values(["fingerprint", "protocol", "fold"])
    .reset_index(drop=True)
)

lr_table

,model,fingerprint,protocol,fold,C,l1_ratio,class_weight,n_nonzero_coefficients,sparsity,l1_norm,l2_norm,inner_pr_auc,test_pr_auc,inner_test_gap,train_test_gap
0,Logistic Regression,ECFP4,OOD holdout,1,0.100,0.00,balanced,1024.0,0.000000,184.214396,7.266730,0.884100,0.9544,-0.070300,0.0394
1,Logistic Regression,ECFP4,OOD holdout,2,0.100,0.00,balanced,1024.0,0.000000,181.681821,7.144868,0.809286,0.9680,-0.158714,-0.3392
2,Logistic Regression,ECFP4,OOD holdout,3,0.100,0.00,NaN,1024.0,0.000000,190.027505,7.515784,0.909542,0.9465,-0.036958,0.0403
3,Logistic Regression,ECFP4,Random shuffle,1,0.100,0.00,NaN,886.0,0.134766,49.889489,2.411850,0.989860,0.5947,0.395160,0.4012
4,Logistic Regression,ECFP4,Random shuffle,2,1.000,0.25,NaN,298.0,0.708984,73.310906,5.707287,0.380320,0.7144,-0.334080,0.2090
5,Logistic Regression,ECFP4,Random shuffle,3,0.005,0.00,NaN,846.0,0.173828,5.789614,0.348388,0.967761,0.6901,0.277661,0.2937
6,Logistic Regression,MACCS,OOD holdout,1,0.100,0.25,balanced,110.0,0.341317,28.725755,3.499203,0.840805,0.8307,0.010105,0.1135
7,Logistic Regression,MACCS,OOD holdout,2,0.500,0.00,balanced,150.0,0.101796,51.612308,5.452131,0.733026,0.8851,-0.152074,-0.6735
8,Logistic Regression,MACCS,OOD holdout,3,0.050,0.50,NaN,77.0,0.538922,16.090594,2.541230,0.788045,0.8135,-0.025455,0.1193
9,Logistic Regression,MACCS,Random shuffle,1,1.000,1.00,NaN,47.0,0.718563,30.954638,6.063429,0.970116,0.5441,0.426016,0.4406


## Linear SVM complexity table

For linear SVM the indicators are:

- selected `C`
- L2 norm of the weight vector
- approximate margin

In [9]:
svm_cols = [
    "model",
    "fingerprint",
    "protocol",
    "fold",
    "C",
    "class_weight",
    "l2_norm",
    "approx_margin",
    "n_nonzero_coefficients",
    "inner_pr_auc",
    "test_pr_auc",
    "inner_test_gap",
    "train_test_gap",
]

svm_table = (
    complexity_all[complexity_all["model_short"] == "SVM"]
    [svm_cols]
    .sort_values(["fingerprint", "protocol", "fold"])
    .reset_index(drop=True)
)

svm_table

,model,fingerprint,protocol,fold,C,class_weight,l2_norm,approx_margin,n_nonzero_coefficients,inner_pr_auc,test_pr_auc,inner_test_gap,train_test_gap
0,SVM linear,ECFP4,OOD holdout,1,0.050,balanced,5.420886,0.184472,1024.0,0.885327,0.9393,-0.053973,0.0523
1,SVM linear,ECFP4,OOD holdout,2,0.050,balanced,5.370060,0.186218,1024.0,0.816601,0.9548,-0.138199,-0.3634
2,SVM linear,ECFP4,OOD holdout,3,0.050,balanced,5.748497,0.173959,1024.0,0.902570,0.9380,-0.035430,0.0329
3,SVM linear,ECFP4,Random shuffle,1,0.010,NaN,0.781926,1.278893,725.0,0.993564,0.6479,0.345664,0.3463
4,SVM linear,ECFP4,Random shuffle,2,0.010,NaN,0.229390,4.359386,576.0,0.397354,0.7522,-0.354846,0.0730
5,SVM linear,ECFP4,Random shuffle,3,0.001,balanced,0.363155,2.753645,842.0,0.968177,0.6829,0.285277,0.2993
6,SVM linear,MACCS,OOD holdout,1,0.050,balanced,3.351561,0.298368,149.0,0.836424,0.8285,0.007924,0.1109
7,SVM linear,MACCS,OOD holdout,2,50.000,balanced,10.872555,0.091975,147.0,0.733004,0.8734,-0.140396,-0.5824
8,SVM linear,MACCS,OOD holdout,3,0.100,NaN,3.937703,0.253955,150.0,0.785255,0.7998,-0.014545,0.1146
9,SVM linear,MACCS,Random shuffle,1,2.000,NaN,7.749767,0.129036,135.0,0.972179,0.5052,0.466979,0.4868


## Decision Tree complexity table

For Decision Trees:

- selected `ccp_alpha`
- selected `max_depth`
- effective tree depth
- number of nodes
- number of leaves
- number of features used in the tree
- average minimum depth of the used features

The number of nodes and leaves gives a direct indication of how large the fitted tree is.

In [10]:
dt_cols = [
    "model",
    "fingerprint",
    "protocol",
    "fold",
    "ccp_alpha",
    "max_depth",
    "effective_depth",
    "n_nodes",
    "n_leaves",
    "n_features_used",
    "used_feature_fraction",
    "feature_min_depth_mean",
    "feature_min_depth_std",
    "inner_pr_auc",
    "test_pr_auc",
    "inner_test_gap",
    "train_test_gap",
]

dt_table = (
    complexity_all[complexity_all["model_short"] == "DT"]
    [dt_cols]
    .sort_values(["fingerprint", "protocol", "fold"])
    .reset_index(drop=True)
)

dt_table

,model,fingerprint,protocol,fold,ccp_alpha,max_depth,effective_depth,n_nodes,n_leaves,n_features_used,used_feature_fraction,feature_min_depth_mean,feature_min_depth_std,inner_pr_auc,test_pr_auc,inner_test_gap,train_test_gap
0,Decision Tree,ECFP4,OOD holdout,1,0.0000,20.0,20.0,985.0,493.0,358.0,0.349609,12.273743,4.217274,0.862853,0.9105,-0.047647,0.0678
1,Decision Tree,ECFP4,OOD holdout,2,0.0010,20.0,20.0,339.0,170.0,152.0,0.148438,10.243421,4.435201,0.785762,0.9438,-0.158038,-0.3859
2,Decision Tree,ECFP4,OOD holdout,3,0.0000,20.0,20.0,551.0,276.0,220.0,0.214844,9.572727,4.198396,0.902256,0.9726,-0.070344,0.0144
3,Decision Tree,ECFP4,Random shuffle,1,0.0000,20.0,20.0,153.0,77.0,72.0,0.070312,7.777778,4.531440,0.993223,0.5623,0.430923,0.4302
4,Decision Tree,ECFP4,Random shuffle,2,0.0100,5.0,5.0,21.0,11.0,10.0,0.009766,2.300000,1.268858,0.404528,0.7389,-0.334372,-0.1587
5,Decision Tree,ECFP4,Random shuffle,3,0.0000,20.0,20.0,73.0,37.0,36.0,0.035156,7.000000,5.238745,0.970758,0.6454,0.325358,0.3343
6,Decision Tree,MACCS,OOD holdout,1,0.0000,15.0,15.0,585.0,293.0,100.0,0.598802,6.380000,2.737809,0.880837,0.9490,-0.068163,0.0428
7,Decision Tree,MACCS,OOD holdout,2,0.0000,10.0,10.0,373.0,187.0,92.0,0.550898,5.956522,2.264629,0.815139,0.9524,-0.137261,-0.3808
8,Decision Tree,MACCS,OOD holdout,3,0.0001,20.0,20.0,1013.0,507.0,120.0,0.718563,6.258333,2.447774,0.854999,0.9355,-0.080501,0.0547
9,Decision Tree,MACCS,Random shuffle,1,0.0000,10.0,10.0,69.0,35.0,29.0,0.173653,5.000000,2.573070,0.969246,0.5277,0.441546,0.4492


## Gap analysis table

This table collects the three main performance levels:

- train PR-AUC
- inner validation PR-AUC
- final OOD test PR-AUC

and the corresponding gaps:

$$\text{train-inner gap} = \text{train PR-AUC} - \text{inner PR-AUC}$$

$$\text{inner-test gap} = \text{inner PR-AUC} - \text{test PR-AUC}$$

$$\text{train-test gap} = \text{train PR-AUC} - \text{test PR-AUC}$$


This table connects model selection, overfitting and OOD generalization.

In [11]:
gap_cols = [
    "model",
    "model_short",
    "fingerprint",
    "protocol",
    "fold",
    "train_pr_auc",
    "inner_pr_auc",
    "inner_train_pr_auc",
    "test_pr_auc",
    "train_inner_gap",
    "inner_test_gap",
    "train_test_gap",
]

gap_analysis = (
    complexity_all[gap_cols]
    .sort_values(["model_short", "fingerprint", "protocol", "fold"])
    .reset_index(drop=True)
)

gap_analysis

,model,model_short,fingerprint,protocol,fold,train_pr_auc,inner_pr_auc,inner_train_pr_auc,test_pr_auc,train_inner_gap,inner_test_gap,train_test_gap
0,Decision Tree,DT,ECFP4,OOD holdout,1,0.9783,0.862853,0.951132,0.9105,0.115447,-0.047647,0.0678
1,Decision Tree,DT,ECFP4,OOD holdout,2,0.5579,0.785762,0.953923,0.9438,-0.227862,-0.158038,-0.3859
2,Decision Tree,DT,ECFP4,OOD holdout,3,0.9870,0.902256,0.982301,0.9726,0.084744,-0.070344,0.0144
3,Decision Tree,DT,ECFP4,Random shuffle,1,0.9925,0.993223,0.993197,0.5623,-0.000723,0.430923,0.4302
4,Decision Tree,DT,ECFP4,Random shuffle,2,0.5802,0.404528,0.671043,0.7389,0.175672,-0.334372,-0.1587
5,Decision Tree,DT,ECFP4,Random shuffle,3,0.9797,0.970758,0.983375,0.6454,0.008942,0.325358,0.3343
6,Decision Tree,DT,MACCS,OOD holdout,1,0.9918,0.880837,0.949371,0.9490,0.110963,-0.068163,0.0428
7,Decision Tree,DT,MACCS,OOD holdout,2,0.5716,0.815139,0.945328,0.9524,-0.243539,-0.137261,-0.3808
8,Decision Tree,DT,MACCS,OOD holdout,3,0.9902,0.854999,0.965732,0.9355,0.135201,-0.080501,0.0547
9,Decision Tree,DT,MACCS,Random shuffle,1,0.9769,0.969246,0.982848,0.5277,0.007654,0.441546,0.4492


## Aggregated complexity summary


In [12]:
summary_metrics = [
    "inner_pr_auc",
    "test_pr_auc",
    "inner_test_gap",
    "train_test_gap",
    "l2_norm",
    "n_nonzero_coefficients",
    "approx_margin",
    "effective_depth",
    "n_nodes",
    "n_leaves",
    "n_features_used",
    "feature_min_depth_mean",
]

available_summary_metrics = [
    col for col in summary_metrics
    if col in complexity_all.columns
]

complexity_summary = (
    complexity_all
    .groupby(["model", "model_short", "fingerprint", "protocol"], as_index=False)
    [available_summary_metrics]
    .agg(["mean", "std"])
)

complexity_summary.columns = [
    "_".join([c for c in col if c])
    for col in complexity_summary.columns
]

complexity_summary = complexity_summary.reset_index()

complexity_summary

,index,model,model_short,fingerprint,protocol,inner_pr_auc_mean,inner_pr_auc_std,test_pr_auc_mean,test_pr_auc_std,inner_test_gap_mean,...,effective_depth_mean,effective_depth_std,n_nodes_mean,n_nodes_std,n_leaves_mean,n_leaves_std,n_features_used_mean,n_features_used_std,feature_min_depth_mean_mean,feature_min_depth_mean_std
0,0,Decision Tree,DT,ECFP4,OOD holdout,0.850290,0.059254,0.942300,0.031077,-0.092010,...,20.000000,0.000000,625.000000,329.296219,313.000000,164.648110,243.333333,104.963486,10.696630,1.406386
1,1,Decision Tree,DT,ECFP4,Random shuffle,0.789503,0.333587,0.648867,0.088351,0.140636,...,15.000000,8.660254,82.333333,66.493107,41.666667,33.246554,39.333333,31.134118,5.692593,2.963697
2,2,Decision Tree,DT,MACCS,OOD holdout,0.850325,0.033097,0.945633,0.008939,-0.095308,...,15.000000,5.000000,657.000000,326.018404,329.000000,163.009202,104.000000,14.422205,6.198285,0.218032
3,3,Decision Tree,DT,MACCS,Random shuffle,0.783690,0.323201,0.606200,0.105175,0.177490,...,13.666667,5.507571,94.333333,58.286648,47.666667,29.143324,36.666667,18.717194,5.580960,1.770517
4,4,Logistic Regression,LR,ECFP4,OOD holdout,0.867643,0.052115,0.956300,0.010875,-0.088657,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,5,Logistic Regression,LR,ECFP4,Random shuffle,0.779314,0.345715,0.666400,0.063272,0.112914,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,6,Logistic Regression,LR,MACCS,OOD holdout,0.787292,0.053893,0.843100,0.037376,-0.055808,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,7,Logistic Regression,LR,MACCS,Random shuffle,0.768142,0.355370,0.610467,0.113568,0.157675,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,8,Logistic Regression,LR,RDKit desc,OOD holdout,0.807319,0.044027,0.877567,0.029450,-0.070247,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,9,Logistic Regression,LR,RDKit desc,Random shuffle,0.799950,0.302165,0.657833,0.078796,0.142117,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
complexity_all.to_csv(OUTPUT_DIR / "complexity_all.csv", index=False)
lr_table.to_csv(OUTPUT_DIR / "complexity_lr.csv", index=False)
svm_table.to_csv(OUTPUT_DIR / "complexity_svm.csv", index=False)
dt_table.to_csv(OUTPUT_DIR / "complexity_dt.csv", index=False)
gap_analysis.to_csv(OUTPUT_DIR / "complexity_gap_analysis.csv", index=False)
complexity_summary.to_csv(OUTPUT_DIR / "complexity_summary.csv", index=False)

print("Saved complexity tables in:")
print(OUTPUT_DIR)

Saved complexity tables in:
/home/f.capria/drug-discovery-lohi/results/results_ood_vs_random_shuffle/hi/kdr
